# 76 - Experimento hibrido TF-IDF + MPNet + HDBSCAN

Este notebook combina a representacao hibrida do notebook 52 com a avaliacao e inspecao dos notebooks 74 e 75. Cada descricao recebe um vetor TF-IDF normalizado e um embedding MPNet normalizado; os blocos sao ponderados por `ALPHA` e concatenados antes do agrupamento.

**Pergunta:** a combinacao de sinais lexicais e semanticos forma grupos densos uteis sob diferentes configuracoes TF-IDF e parametros HDBSCAN?

A analise usa descricoes normalizadas unicas, sem NCM ou rotulos humanos, e contabiliza ruido explicitamente. A grade cruza cinco analyzers TF-IDF com nove configuracoes HDBSCAN (45 ajustes). O custo pode ser significativo.


## Configuracoes e comparabilidade

Controles comuns aos notebooks 70 a 76:

- Base completa `2025_NCMs_filtrados_semstopwords`, periodo 202501-202512.
- Ate 20.000 descricoes unicas para agrupamento, amostradas com semente 42 e ordenadas por descricao normalizada. Frequencia nao pondera o ajuste.
- Silhouette cosseno sobre uma amostra base fixa de ate 20.000 linhas, com semente 42. Ruido do algoritmo e excluido depois da amostragem; a quantidade efetivamente avaliada e registrada.
- Ranking exige silhouette definida e de 10 a 1.000 clusters efetivamente formados, sem contar ruido. Pesos: 0,75 para silhouette reescalada e 0,25 para cobertura do algoritmo.
- t-SNE 2D e 3D: mesmos ate 5.000 itens, semente 42, TruncatedSVD de ate 50 componentes, perplexidade 30, 1.000 iteracoes, init PCA (quando ha dimensoes suficientes), learning_rate auto e Barnes-Hut. As projecoes de representacoes diferentes sao exploratorias, nao um espaco comum de distancias.
- `comparison` em `run.json` registra os controles e o hash das descricoes/frequencias usadas no agrupamento.

TF-IDF: os mesmos 10 analyzers dos experimentos de texto, cruzados com `MIN_DF_VALUES = [1, 2, 5, 10]`, max_features 100.000, max_df 1.0, sublinear_tf True, norm l2 e dtype float64. Os vetores zerados sao registrados como diagnostico da representacao, separados do ruido do algoritmo.

MPNet: `sentence-transformers/paraphrase-multilingual-mpnet-base-v2`, batch_size 64, normalize_embeddings True e saida float32. Nao ha min_df ou analyzer no MPNet puro.

O hibrido usa alpha 0,2 como fator especifico da representacao, com normalizacao l2 por bloco e pesos sqrt(alpha)/sqrt(1-alpha). A grade HDBSCAN coincide com 74 e 75: 40 representacoes TF-IDF x 16 combinacoes = 640 ajustes. Os filtros extras de silhouette e o fallback ficam desativados para a comparacao padrao.


In [ ]:
from sklearn import config_context
from hashlib import sha256
from datetime import datetime, timezone
from pathlib import Path
import json
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import sparse
from sentence_transformers import SentenceTransformer
from sklearn.cluster import HDBSCAN
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import normalize

from nofis_classifier.schema import DESCRIPTION_NORMALIZED, FREQUENCY

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("notebook76")
pd.set_option("display.max_colwidth", 140)

DATASET_NAME = "2025_NCMs_filtrados_semstopwords"
PROJECT_ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "data" / "processed").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Nao encontrei a raiz do repositorio com data/processed/.")
DATASET_DIR = PROJECT_ROOT / "data" / "processed" / DATASET_NAME
DATASET_FILES = sorted(DATASET_DIR.glob("*.parquet"))
if not DATASET_FILES:
    raise FileNotFoundError(f"Nenhum Parquet encontrado para {DATASET_NAME}: {DATASET_DIR}")
if len(DATASET_FILES) > 1:
    raise ValueError(
        f"Mais de um Parquet encontrado em {DATASET_DIR}; selecione o periodo explicitamente: "
        f"{[path.name for path in DATASET_FILES]}"
    )
DATASET_PATH = DATASET_FILES[0]
if not DATASET_PATH.stem.startswith(f"{DATASET_NAME}-"):
    raise ValueError(f"Nome de arquivo inesperado: {DATASET_PATH.name}")
PERIOD_LABEL = DATASET_PATH.stem[len(DATASET_NAME) + 1:]

# Parametros de representacao do notebook 52.
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
MAX_DESCRIPTIONS = 20_000
SAMPLE_RANDOM_STATE = 42
ENCODE_BATCH_SIZE = 64
ANALYZER_CONFIGS = {'char_wb_2_4': ('char_wb', (2, 4)), 'char_wb_2_10': ('char_wb', (2, 10)), 'char_wb_3_5': ('char_wb', (3, 5)), 'char_wb_3_10': ('char_wb', (3, 10)), 'char_wb_4_8': ('char_wb', (4, 8)), 'char_wb_4_10': ('char_wb', (4, 10)), 'char_wb_5_10': ('char_wb', (5, 10)), 'word_1_3': ('word', (1, 3)), 'word_1_5': ('word', (1, 5)), 'word_2_5': ('word', (2, 5))}
ALPHA = 0.2

# Grade HDBSCAN e avaliacao dos notebooks 74 e 75.
MIN_CLUSTER_SIZE_VALUES = [2, 5, 10, 20]
MIN_SAMPLES_VALUES = [None, 2, 5, 10]
CLUSTER_SELECTION_METHOD = "eom"
HDBSCAN_N_JOBS = 1
SILHOUETTE_SAMPLE_SIZE = 20_000
MAX_CLUSTERS_FOR_RANKING = 1000
MIN_CLUSTER_FOR_RANKING = 10
SILHOUETTE_WEIGHT = 0.75
NOISE_WEIGHT = 0.25
TSNE_MAX_POINTS = 5_000
TSNE_PERPLEXITY = 30
TSNE_MAX_ITER = 1_000
TSNE_RANDOM_STATE = 42
TSNE_SVD_COMPONENTS = 50
TSNE_3D_N_COMPONENTS = 3
RANKING_FILTER_SILHOUETTE_ABOVE_NOISE = False
SHOW_FULL_BASE_IF_RANKING_EMPTY = False
SELECTED_RUN_ID = None  # None seleciona o primeiro run do ranking filtrado.
OUTPUT_ROOT = PROJECT_ROOT / "reports/tfidf_mpnet_hdbscan"
MODEL_ROOT = PROJECT_ROOT / "models/tfidf_mpnet_hdbscan"

METRIC_RANDOM_STATE = 42

PAIRWISE_WORKING_MEMORY_MIB = 64

MIN_DF_VALUES = [1, 2, 5, 10]

MAX_FEATURES = 100_000

MAX_DF = 1.0

ARTIFACT_RUN_ID = None


## 2. Funcoes da analise

As funcoes mantem no notebook a preparacao, a combinacao das representacoes, a avaliacao e a inspecao dos clusters. A concatenacao preserva TF-IDF esparso e valida a ordem das linhas.


In [ ]:
def prepare_items(frame):
    """Limpa descricoes, agrega duplicatas e preserva a frequencia total."""
    initial_frequency = frame[FREQUENCY].sum()
    prepared = frame[[DESCRIPTION_NORMALIZED, FREQUENCY]].copy()
    prepared[DESCRIPTION_NORMALIZED] = prepared[DESCRIPTION_NORMALIZED].astype("string").str.strip()
    prepared = prepared[prepared[DESCRIPTION_NORMALIZED].notna() & prepared[DESCRIPTION_NORMALIZED].ne("")]
    prepared = prepared[prepared[FREQUENCY] >= 1].copy()
    prepared = prepared.groupby(DESCRIPTION_NORMALIZED, as_index=False, sort=False)[FREQUENCY].sum()
    if prepared[FREQUENCY].sum() != initial_frequency:
        raise ValueError("A preparacao alterou a soma de frequencia.")
    return prepared.sort_values(DESCRIPTION_NORMALIZED).reset_index(drop=True)


def combine_representations(tfidf_matrix, mpnet_matrix, alpha, tfidf_ids, mpnet_ids):
    """Normaliza e concatena TF-IDF e MPNet com pesos controlados por alpha."""
    try:
        alpha = float(alpha)
    except (TypeError, ValueError) as exc:
        raise ValueError("alpha deve ser um numero finito no intervalo [0, 1].") from exc
    if not np.isfinite(alpha) or not 0.0 <= alpha <= 1.0:
        raise ValueError("alpha deve ser um numero finito no intervalo [0, 1].")
    if tfidf_matrix.ndim != 2 or mpnet_matrix.ndim != 2:
        raise ValueError("As representacoes devem ser matrizes bidimensionais.")
    if tfidf_matrix.shape[0] != mpnet_matrix.shape[0]:
        raise ValueError("As representacoes devem ter a mesma quantidade de linhas.")
    tfidf_ids, mpnet_ids = np.asarray(tfidf_ids), np.asarray(mpnet_ids)
    rows = tfidf_matrix.shape[0]
    if len(tfidf_ids) != rows or len(mpnet_ids) != rows:
        raise ValueError("Cada matriz deve ter um identificador por linha.")
    if len(np.unique(tfidf_ids)) != rows or len(np.unique(mpnet_ids)) != rows:
        raise ValueError("Os identificadores devem ser unicos em cada representacao.")
    if not np.array_equal(tfidf_ids, mpnet_ids):
        raise ValueError("A ordem dos registros difere entre TF-IDF e MPNet.")
    if alpha == 1.0:
        return normalize(tfidf_matrix, norm="l2", axis=1, copy=True)
    if alpha == 0.0:
        return normalize(mpnet_matrix, norm="l2", axis=1, copy=True)
    tfidf_normalized = normalize(tfidf_matrix, norm="l2", axis=1, copy=True)
    mpnet_normalized = normalize(mpnet_matrix, norm="l2", axis=1, copy=True)
    tfidf_weighted = tfidf_normalized * np.sqrt(alpha)
    mpnet_weighted = mpnet_normalized * np.sqrt(1.0 - alpha)
    if sparse.issparse(tfidf_weighted):
        return sparse.hstack((tfidf_weighted, sparse.csr_matrix(mpnet_weighted)), format="csr")
    return np.concatenate((tfidf_weighted, mpnet_weighted), axis=1)


def summarize_clusters(items, labels):
    """Retorna atribuicoes e resumo de tamanho e frequencia por cluster."""
    result = items.copy()
    result["cluster"] = np.asarray(labels)
    result["ruido"] = result["cluster"].eq(-1)
    summary = (
        result.groupby("cluster", as_index=False)
        .agg(descricoes_unicas=(DESCRIPTION_NORMALIZED, "size"), frequencia_total=(FREQUENCY, "sum"))
        .sort_values("descricoes_unicas", ascending=False)
    )
    return result, summary


def score_configuration(matrix, labels, sample_size, random_state):
    """Avalia a mesma amostra de linhas, excluindo ruido somente apos amostrar."""
    labels = np.asarray(labels)
    noise_mask = labels == -1
    cluster_count = len(np.unique(labels[~noise_mask]))
    base_count = min(sample_size, len(labels))
    rng = np.random.default_rng(random_state)
    positions = np.sort(rng.choice(len(labels), size=base_count, replace=False))
    positions = positions[~noise_mask[positions]]
    sample_labels = labels[positions]
    sample_cluster_count = len(np.unique(sample_labels))
    score = np.nan
    actual_count = len(positions)
    if 2 <= sample_cluster_count < actual_count:
        with config_context(working_memory=PAIRWISE_WORKING_MEMORY_MIB):
            score = float(silhouette_score(matrix[positions], sample_labels, metric="cosine"))
    else:
        actual_count = 0
    return {
        "cluster_count_excluding_noise": cluster_count,
        "noise_count": int(noise_mask.sum()),
        "noise_fraction": float(noise_mask.mean()),
        "silhouette_base_sample_size": base_count,
        "silhouette_sample_size": actual_count,
        "silhouette_cosine_without_noise": score,
    }


def compute_tsne_projection(features, max_points, perplexity, max_iter, random_state, n_components=2):
    """Usa a mesma amostra, SVD e parametros t-SNE em todas as representacoes."""
    point_count = features.shape[0]
    sample_count = min(point_count, max_points)
    if sample_count < 3 or n_components not in (2, 3):
        raise ValueError("t-SNE exige pelo menos tres itens e n_components 2 ou 3.")
    rng = np.random.default_rng(random_state)
    indices = np.sort(rng.choice(point_count, size=sample_count, replace=False))
    sampled = features[indices]
    dimensions = min(TSNE_SVD_COMPONENTS, sampled.shape[1] - 1, sample_count - 1)
    if dimensions >= 1:
        reduced = TruncatedSVD(n_components=dimensions, random_state=random_state).fit_transform(sampled)
    else:
        reduced = sampled.toarray() if hasattr(sampled, "toarray") else np.asarray(sampled)
        dimensions = reduced.shape[1]
    coordinates = TSNE(
        n_components=n_components,
        perplexity=min(float(perplexity), (sample_count - 1) / 3),
        init="pca" if dimensions >= n_components else "random",
        learning_rate="auto", max_iter=max_iter, random_state=random_state,
        method="barnes_hut",
    ).fit_transform(reduced)
    return indices, coordinates, dimensions

def save_tsne_plot(coordinates, labels, title, output_path):
    """Salva uma visualizacao t-SNE 2D colorida pelas atribuicoes HDBSCAN."""
    labels = np.asarray(labels)
    figure, axis = plt.subplots(figsize=(12, 9))
    noise = labels == -1
    axis.scatter(coordinates[noise, 0], coordinates[noise, 1], s=10, c="#999999", alpha=0.35, label="Ruido (-1)")
    cluster_labels = np.unique(labels[~noise])
    if len(cluster_labels):
        colors = plt.get_cmap("turbo", max(len(cluster_labels), 1))
        for position, cluster in enumerate(cluster_labels):
            mask = labels == cluster
            axis.scatter(coordinates[mask, 0], coordinates[mask, 1], s=12, color=colors(position), alpha=0.75, label=f"Cluster {cluster}")
    axis.set(title=title, xlabel="t-SNE 1", ylabel="t-SNE 2")
    if len(cluster_labels) <= 15:
        axis.legend(loc="best", fontsize="small")
    figure.tight_layout()
    figure.savefig(output_path, dpi=160)
    plt.close(figure)



def save_tsne_3d_html(coordinates, labels, run_id, output_path):
    """Salva uma visualizacao t-SNE 3D interativa e autocontida em HTML."""
    points = [
        {"x": float(point[0]), "y": float(point[1]), "z": float(point[2]), "label": int(label)}
        for point, label in zip(coordinates, labels)
    ]
    points_json = json.dumps(points, separators=(",", ":"), ensure_ascii=False).replace("<", "\\u003c")
    title = json.dumps(str(run_id), ensure_ascii=False).replace("<", "\\u003c")
    page = r"""<!doctype html>
<html lang="pt-BR"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>t-SNE 3D</title><style>
html,body{margin:0;height:100%;font:14px system-ui,sans-serif;color:#202124;background:#f7f8fa}main{display:flex;flex-direction:column;height:100%}header{padding:12px 18px;background:white;border-bottom:1px solid #ddd}h1{margin:0 0 4px;font-size:17px}p{margin:0;color:#555}canvas{display:block;width:100%;flex:1;min-height:300px;cursor:grab}canvas:active{cursor:grabbing}
</style></head><body><main><header><h1 id="title"></h1><p>Arraste para girar; use a roda do mouse para aproximar. Ruido em cinza.</p></header><canvas id="plot"></canvas></main>
<script>const points=__POINTS__,runTitle=__TITLE__,canvas=document.getElementById("plot"),ctx=canvas.getContext("2d");document.getElementById("title").textContent="t-SNE 3D: "+runTitle;
let yaw=.65,pitch=-.35,zoom=1,dragging=false,lastX=0,lastY=0;
const colors=["#3366cc","#dc3912","#ff9900","#109618","#990099","#0099c6","#dd4477","#66aa00","#b82e2e","#316395","#994499","#22aa99","#aaaa11","#6633cc","#e67300","#8b0707","#651067","#329262","#5574a6","#3b3eac"];
const bounds=["x","y","z"].map(k=>{const a=points.map(p=>p[k]),lo=Math.min(...a),hi=Math.max(...a);return{lo,span:Math.max(hi-lo,1e-9)}});
function resize(){const r=devicePixelRatio||1;canvas.width=Math.max(1,Math.floor(canvas.clientWidth*r));canvas.height=Math.max(1,Math.floor(canvas.clientHeight*r));ctx.setTransform(r,0,0,r,0,0);draw()}
function draw(){const w=canvas.clientWidth,h=canvas.clientHeight;ctx.clearRect(0,0,w,h);const scale=Math.min(w,h)*.7*zoom,cy=Math.cos(yaw),sy=Math.sin(yaw),cp=Math.cos(pitch),sp=Math.sin(pitch);const projected=points.map(p=>{const x=((p.x-bounds[0].lo)/bounds[0].span-.5)*2,y=((p.y-bounds[1].lo)/bounds[1].span-.5)*2,z=((p.z-bounds[2].lo)/bounds[2].span-.5)*2,rx=x*cy-z*sy,rz=x*sy+z*cy,ry=y*cp-rz*sp,d=y*sp+rz*cp;return{x:w/2+rx*scale/2,y:h/2-ry*scale/2,d,label:p.label}}).sort((a,b)=>a.d-b.d);for(const p of projected){ctx.globalAlpha=p.label === -1 ? 0.24 : 0.78;ctx.fillStyle=p.label===-1?"#999999":colors[Math.abs(p.label)%colors.length];ctx.beginPath();ctx.arc(p.x,p.y,p.label === -1 ? 2 : 3,0,Math.PI*2);ctx.fill()}ctx.globalAlpha=1}
canvas.addEventListener("pointerdown",e=>{dragging=true;lastX=e.clientX;lastY=e.clientY;canvas.setPointerCapture(e.pointerId)});canvas.addEventListener("pointerup",()=>dragging=false);canvas.addEventListener("pointermove",e=>{if(!dragging)return;yaw+=(e.clientX-lastX)*.008;pitch=Math.max(-1.5,Math.min(1.5,pitch+(e.clientY-lastY)*.008));lastX=e.clientX;lastY=e.clientY;draw()});canvas.addEventListener("wheel",e=>{e.preventDefault();zoom=Math.max(.3,Math.min(5,zoom*(e.deltaY<0?1.1:.9)));draw()},{passive:false});window.addEventListener("resize",resize);resize();</script></body></html>"""
    page = page.replace("__POINTS__", points_json).replace("__TITLE__", title)
    output_path.write_text(page, encoding="utf-8")


def comparison_metadata(frame):
    """Registra controles comuns e a identidade exata da amostra de agrupamento."""
    sample = frame[[DESCRIPTION_NORMALIZED, FREQUENCY]].sort_values(DESCRIPTION_NORMALIZED).reset_index(drop=True)
    return {
        "dataset_name": DATASET_NAME,
        "training_description_count": len(sample),
        "training_sample_sha256": sha256(pd.util.hash_pandas_object(sample, index=False).values.tobytes()).hexdigest(),
        "max_descriptions": MAX_DESCRIPTIONS, "sample_seed": 42,
        "frequency_weighted": False,
        "silhouette_metric": "cosine", "silhouette_sample_size": SILHOUETTE_SAMPLE_SIZE,
        "metric_random_state": METRIC_RANDOM_STATE,
        "metric_sampling": "fixed rows, then exclude algorithmic noise",
        "min_cluster_for_ranking": MIN_CLUSTER_FOR_RANKING,
        "max_clusters_for_ranking": MAX_CLUSTERS_FOR_RANKING,
        "silhouette_weight": SILHOUETTE_WEIGHT, "noise_weight": NOISE_WEIGHT,
        "tsne": {
            "max_points": TSNE_MAX_POINTS, "perplexity": TSNE_PERPLEXITY,
            "max_iter": TSNE_MAX_ITER, "random_state": TSNE_RANDOM_STATE,
            "reduction": "TruncatedSVD", "svd_components": TSNE_SVD_COMPONENTS,
            "dimensions": [2, 3], "init": "pca", "learning_rate": "auto",
            "method": "barnes_hut",
        },
    }


## 3. Carregar e preparar os dados

Origem: Parquet de data/processed/2025_NCMs_filtrados_semstopwords/. Usa a mesma amostra de ate 20.000 descricoes e a mesma ordenacao dos demais experimentos.


In [ ]:
raw_items = pd.read_parquet(DATASET_PATH)
required_columns = {DESCRIPTION_NORMALIZED, FREQUENCY}
missing_columns = required_columns - set(raw_items.columns)
if missing_columns:
    raise ValueError(f"Colunas ausentes no dataset: {sorted(missing_columns)}")
base_items = prepare_items(raw_items)
logger.info("Descricoes unicas disponiveis: %d", len(base_items))
sampled_items = base_items
if MAX_DESCRIPTIONS is not None and len(base_items) > MAX_DESCRIPTIONS:
    sampled_items = base_items.sample(n=MAX_DESCRIPTIONS, random_state=SAMPLE_RANDOM_STATE)
sampled_items = sampled_items.sort_values(DESCRIPTION_NORMALIZED).reset_index(drop=True)
sampled_items.insert(0, "descricao_id", np.arange(len(sampled_items), dtype=np.int64))
if sampled_items.empty:
    raise ValueError("Nao ha descricoes validas para o experimento.")
display(sampled_items.head())
logger.info("Descricoes na amostra: %d", len(sampled_items))


## 4. Gerar embeddings MPNet

MPNet e calculado uma vez e reutilizado nas configuracoes TF-IDF. Os embeddings sao normalizados na funcao de combinacao, conforme o notebook 52.


In [ ]:
texts = sampled_items[DESCRIPTION_NORMALIZED].astype(str).tolist()
row_ids = sampled_items["descricao_id"].to_numpy()
logger.info("Carregando modelo %s", MODEL_NAME)
encoder = SentenceTransformer(MODEL_NAME)
mpnet_matrix = encoder.encode(
    texts, batch_size=ENCODE_BATCH_SIZE, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=True,
).astype(np.float32, copy=False)
if mpnet_matrix.shape[0] != len(sampled_items):
    raise ValueError("MPNet nao preservou uma linha por descricao.")
logger.info("Embeddings MPNet: %d x %d", *mpnet_matrix.shape)


## 5. Ajustar TF-IDF + HDBSCAN

Cruza os mesmos analyzers e min_df de 70, 72 e 74 com a grade HDBSCAN de 74 e 75. MPNet e reutilizado. O alpha permanece como fator especifico do hibrido. Silhouette usa os controles comuns de amostragem, excluindo ruido apos selecionar as linhas.


In [ ]:
records = []
labels_by_run = {}
items_by_run = {}
hybrid_matrices = {}
vectorizers = {}

for config_name, (analyzer, ngram_range) in ANALYZER_CONFIGS.items():
    for min_df in MIN_DF_VALUES:
        representation_key = f"{config_name}_min_df{min_df}"
        logger.info("Ajustando TF-IDF: %s (%s, %s)", config_name, analyzer, ngram_range)
        vectorizer = TfidfVectorizer(
            analyzer=analyzer, ngram_range=ngram_range, min_df=min_df,
            max_features=MAX_FEATURES, max_df=MAX_DF, sublinear_tf=True, norm="l2", dtype=np.float64,
        )
        tfidf_matrix = vectorizer.fit_transform(texts)
        if tfidf_matrix.shape[0] != len(sampled_items):
            raise ValueError(f"TF-IDF ({config_name}) nao preservou uma linha por descricao.")
        hybrid_matrix = combine_representations(tfidf_matrix, mpnet_matrix, ALPHA, row_ids, row_ids.copy())
        hybrid_matrices[representation_key] = hybrid_matrix
        vectorizers[representation_key] = vectorizer
        logger.info("%s: TF-IDF=%d features; hibrida=%d dimensoes; %d valores", config_name, tfidf_matrix.shape[1], hybrid_matrix.shape[1], hybrid_matrix.nnz)

        for min_cluster_size in MIN_CLUSTER_SIZE_VALUES:
            for min_samples in MIN_SAMPLES_VALUES:
                min_samples_label = "default" if min_samples is None else str(min_samples)
                run_id = (f"seed{SAMPLE_RANDOM_STATE}_{representation_key}_alpha{ALPHA:g}"
                          f"_min_cluster_size{min_cluster_size}_min_samples{min_samples_label}_{CLUSTER_SELECTION_METHOD}")
                logger.info("Ajustando HDBSCAN %s", run_id)
                model = HDBSCAN(
                    min_cluster_size=min_cluster_size, min_samples=min_samples,
                    metric="cosine", algorithm="brute",
                    cluster_selection_method=CLUSTER_SELECTION_METHOD,
                    n_jobs=HDBSCAN_N_JOBS, copy=True,
                )
                labels = model.fit_predict(hybrid_matrix)
                metrics = score_configuration(hybrid_matrix, labels, SILHOUETTE_SAMPLE_SIZE, METRIC_RANDOM_STATE)
                records.append({
                    "run_id": run_id, "sample_seed": SAMPLE_RANDOM_STATE,
                    "representation": "tfidf_mpnet_hybrid", "tfidf_config": representation_key, "analyzer_config": config_name, "min_df": min_df,
                    "analyzer": analyzer, "ngram_range": str(ngram_range), "alpha": ALPHA,
                    "tfidf_features": tfidf_matrix.shape[1], "mpnet_dimensions": mpnet_matrix.shape[1],
                    "hybrid_dimensions": hybrid_matrix.shape[1], "model_name": MODEL_NAME,
                    "min_cluster_size": min_cluster_size, "min_samples": min_samples,
                    "cluster_selection_method": CLUSTER_SELECTION_METHOD,
                    "frequency_weighted": False, **metrics,
                })
                labels_by_run[run_id] = labels
                items_by_run[run_id] = sampled_items

results = pd.DataFrame(records)
logger.info("Configuracoes HDBSCAN avaliadas: %d", len(results))


## 6. Filtrar e comparar configuracoes

O score combina silhouette reescalada e cobertura com os pesos configurados no inicio. O ranking aplica o intervalo de quantidade de clusters e, por padrao, exige silhouette valida maior que a fracao de ruido. Esses filtros afetam apenas a ordenacao e selecao, nao os ajustes.

Se nenhum resultado passar pelo filtro de silhouette, `SHOW_FULL_BASE_IF_RANKING_EMPTY` permite exibir as configuracoes dentro do intervalo de clusters. Os limites minimo e maximo continuam obrigatorios nesse fallback.

`MIN_CLUSTER_FOR_RANKING = 10` define o minimo do ranking. O intervalo ate `MAX_CLUSTERS_FOR_RANKING` inclui os dois extremos e nao altera os ajustes da grade. Em DBSCAN/HDBSCAN, a contagem exclui o grupo de ruido (-1).


In [ ]:
if not 1 <= MIN_CLUSTER_FOR_RANKING <= MAX_CLUSTERS_FOR_RANKING:
    raise ValueError(
        "Os limites devem atender a 1 <= MIN_CLUSTER_FOR_RANKING <= MAX_CLUSTERS_FOR_RANKING."
    )

if SILHOUETTE_WEIGHT < 0 or NOISE_WEIGHT < 0 or SILHOUETTE_WEIGHT + NOISE_WEIGHT <= 0:
    raise ValueError("Os pesos devem ser nao negativos e pelo menos um deve ser maior que zero.")
results["silhouette_quality"] = (results["silhouette_cosine_without_noise"] + 1) / 2
results["noise_quality"] = 1 - results["noise_fraction"]
weight_total = SILHOUETTE_WEIGHT + NOISE_WEIGHT
results["composite_score"] = (SILHOUETTE_WEIGHT * results["silhouette_quality"] + NOISE_WEIGHT * results["noise_quality"]) / weight_total
cluster_count_mask = results["cluster_count_excluding_noise"].between(
    MIN_CLUSTER_FOR_RANKING, MAX_CLUSTERS_FOR_RANKING, inclusive="both",
)
ranking_mask = cluster_count_mask.copy()
if RANKING_FILTER_SILHOUETTE_ABOVE_NOISE:
    ranking_mask &= (
        results["silhouette_cosine_without_noise"].notna()
        & results["silhouette_cosine_without_noise"].gt(results["noise_fraction"])
    )
ranking = (
    results.loc[ranking_mask]
    .sort_values(
        ["composite_score", "silhouette_cosine_without_noise", "noise_fraction"],
        ascending=[False, False, True], na_position="last",
    )
    .reset_index(drop=True)
)
RANKING_USED_FULL_BASE = False
if ranking.empty and SHOW_FULL_BASE_IF_RANKING_EMPTY:
    logger.warning(
        "O filtro de silhouette nao selecionou resultados; exibindo as configuracoes dentro do intervalo de clusters."
    )
    ranking = (
        results.loc[cluster_count_mask].sort_values(
            ["composite_score", "silhouette_cosine_without_noise", "noise_fraction"],
            ascending=[False, False, True], na_position="last",
        )
        .reset_index(drop=True)
    )
    RANKING_USED_FULL_BASE = True
if ranking.empty:
    raise ValueError(
        f"Nenhum resultado elegivel entre {MIN_CLUSTER_FOR_RANKING} e "
        f"{MAX_CLUSTERS_FOR_RANKING} clusters. Ajuste a grade, os limites "
        "ou o filtro de silhouette."
    )
ranking.insert(0, "rank", range(1, len(ranking) + 1))

if SELECTED_RUN_ID is None:
    SELECTED_RUN_ID = ranking.iloc[0]["run_id"]
if SELECTED_RUN_ID not in labels_by_run:
    raise ValueError(f"SELECTED_RUN_ID nao encontrado: {SELECTED_RUN_ID}")
display(ranking)


## 7. Salvar metricas e todos os t-SNE

Esta celula salva metricas e ranking, coordenadas 2D por configuracao TF-IDF e os graficos 2D/3D para todos os runs. Ela nao salva assignments nem modelo. Use a celula seguinte para salvar esses artifacts de um `ARTIFACT_RUN_ID` especifico, sem recalcular os t-SNE.


In [ ]:
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ%f")
output_dir = OUTPUT_ROOT / f"{PERIOD_LABEL}_{RUN_ID}"
output_dir.mkdir(parents=True, exist_ok=False)
results.to_csv(output_dir / "metrics.csv", index=False, encoding="utf-8-sig")
ranking.to_csv(output_dir / "ranking.csv", index=False, encoding="utf-8-sig")

# Calcula as projecoes para cada matriz e reutiliza cada uma nos nove runs HDBSCAN.
tsne_by_config = {}
tsne_3d_by_config = {}
for config_name, matrix in hybrid_matrices.items():
    logger.info("Calculando t-SNE 2D para %s", config_name)
    indices_2d, coordinates_2d, dimensions_2d = compute_tsne_projection(
        matrix, TSNE_MAX_POINTS, TSNE_PERPLEXITY, TSNE_MAX_ITER,
        TSNE_RANDOM_STATE, n_components=2,
    )
    logger.info("Calculando t-SNE 3D para %s", config_name)
    indices_3d, coordinates_3d, dimensions_3d = compute_tsne_projection(
        matrix, TSNE_MAX_POINTS, TSNE_PERPLEXITY, TSNE_MAX_ITER,
        TSNE_RANDOM_STATE, n_components=TSNE_3D_N_COMPONENTS,
    )
    if not np.array_equal(indices_2d, indices_3d):
        raise ValueError("As projecoes 2D e 3D precisam usar a mesma amostra.")
    tsne_by_config[config_name] = {
        "indices": indices_2d, "coordinates": coordinates_2d,
        "dimensions": dimensions_2d,
    }
    tsne_3d_by_config[config_name] = {
        "indices": indices_3d, "coordinates": coordinates_3d,
        "dimensions": dimensions_3d,
    }

coordinates_dir = output_dir / "tsne_coordinates"
plots_dir = output_dir / "tsne_plots"
html_3d_dir = output_dir / "tsne_3d_html"
coordinates_dir.mkdir()
plots_dir.mkdir()
html_3d_dir.mkdir()
tsne_artifacts = {}
for config_name, projection in tsne_by_config.items():
    indices = projection["indices"]
    frame = sampled_items.iloc[indices][
        [DESCRIPTION_NORMALIZED, FREQUENCY]
    ].reset_index(drop=True)
    frame["tsne_1"] = projection["coordinates"][:, 0]
    frame["tsne_2"] = projection["coordinates"][:, 1]
    coordinate_path = coordinates_dir / f"{config_name}_coordinates.csv"
    frame.to_csv(coordinate_path, index=False, encoding="utf-8-sig")
    tsne_artifacts[f"coordinates:{config_name}"] = str(
        coordinate_path.relative_to(output_dir)
    )

for run_id, labels in labels_by_run.items():
    config_name = results.loc[results["run_id"].eq(run_id), "tfidf_config"].iloc[0]
    projection = tsne_by_config[config_name]
    projection_3d = tsne_3d_by_config[config_name]
    labels_2d = np.asarray(labels)[projection["indices"]]
    labels_3d = np.asarray(labels)[projection_3d["indices"]]
    plot_path = plots_dir / f"{run_id}.png"
    save_tsne_plot(projection["coordinates"], labels_2d, run_id, plot_path)
    html_path = html_3d_dir / f"{run_id}.html"
    save_tsne_3d_html(projection_3d["coordinates"], labels_3d, run_id, html_path)
    tsne_artifacts[f"plot_2d:{run_id}"] = str(plot_path.relative_to(output_dir))
    tsne_artifacts[f"plot_3d:{run_id}"] = str(html_path.relative_to(output_dir))

run_metadata = {
    "comparison": comparison_metadata(sampled_items),
    "dataset_name": DATASET_NAME,
    "dataset_path": str(DATASET_PATH.relative_to(PROJECT_ROOT)),
    "period": PERIOD_LABEL,
    "max_descriptions": MAX_DESCRIPTIONS,
    "sample_seed": SAMPLE_RANDOM_STATE,
    "representation": "tfidf_mpnet_hybrid",
    "alpha": ALPHA,
    "model_name": MODEL_NAME,
    "analyzer_configs": {
        name: {"analyzer": analyzer, "ngram_range": list(ngram_range)}
        for name, (analyzer, ngram_range) in ANALYZER_CONFIGS.items()
    },
    "min_df_values": MIN_DF_VALUES,
    "max_features": MAX_FEATURES, "max_df": MAX_DF,
    "min_cluster_size_values": MIN_CLUSTER_SIZE_VALUES,
    "min_samples_values": MIN_SAMPLES_VALUES,
    "cluster_selection_method": CLUSTER_SELECTION_METHOD,
    "ranking_filter_silhouette_above_noise": RANKING_FILTER_SILHOUETTE_ABOVE_NOISE,
    "show_full_base_if_ranking_empty": SHOW_FULL_BASE_IF_RANKING_EMPTY,
    "ranking_used_full_base": RANKING_USED_FULL_BASE,
    "selected_run_id": SELECTED_RUN_ID,
    "min_cluster_for_ranking": MIN_CLUSTER_FOR_RANKING,
    "max_clusters_for_ranking": MAX_CLUSTERS_FOR_RANKING,
    "output_dir": str(output_dir.relative_to(PROJECT_ROOT)),
    "tsne": {
        "max_points": TSNE_MAX_POINTS,
        "perplexity": TSNE_PERPLEXITY,
        "max_iter": TSNE_MAX_ITER,
        "random_state": TSNE_RANDOM_STATE,
        "svd_components": TSNE_SVD_COMPONENTS,
        "three_dimensional_components": TSNE_3D_N_COMPONENTS,
        "artifacts": tsne_artifacts,
    },
    "frequency_weighted": False,
}
(output_dir / "run.json").write_text(
    json.dumps(run_metadata, indent=2, ensure_ascii=False), encoding="utf-8"
)
logger.info("Metricas e todos os graficos t-SNE salvos em %s", output_dir.relative_to(PROJECT_ROOT))


## 8. Salvar assignments e modelo de um run especifico

Informe o `ARTIFACT_RUN_ID` e execute somente a celula de codigo abaixo. Ela salva o CSV de assignments e o modelo HDBSCAN junto do vetorizador TF-IDF, encoder MPNet e configuracao. Esta celula nao calcula nem salva t-SNE.


In [ ]:
ARTIFACT_RUN_ID = ARTIFACT_RUN_ID or SELECTED_RUN_ID

if ARTIFACT_RUN_ID not in labels_by_run:
    raise ValueError(
        f"ARTIFACT_RUN_ID nao encontrado. Runs disponiveis: {list(labels_by_run)}"
    )

artifact_timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ%f")
artifact_folder = f"{PERIOD_LABEL}_{artifact_timestamp}_{ARTIFACT_RUN_ID}"
assignment_output_dir = OUTPUT_ROOT / f"assignments_{artifact_folder}"
model_output_dir = MODEL_ROOT / artifact_folder
assignment_output_dir.mkdir(parents=True, exist_ok=False)
model_output_dir.mkdir(parents=True, exist_ok=False)

artifact_items = items_by_run[ARTIFACT_RUN_ID]
artifact_labels = labels_by_run[ARTIFACT_RUN_ID]
assignments, artifact_cluster_summary = summarize_clusters(artifact_items, artifact_labels)
assignment_path = assignment_output_dir / f"assignments_{ARTIFACT_RUN_ID}.csv"
assignments.to_csv(assignment_path, index=False, encoding="utf-8-sig")

artifact_run = results.set_index("run_id").loc[ARTIFACT_RUN_ID]
config_name = artifact_run["tfidf_config"]
min_samples = None if pd.isna(artifact_run["min_samples"]) else int(artifact_run["min_samples"])
saved_model = HDBSCAN(
    min_cluster_size=int(artifact_run["min_cluster_size"]),
    min_samples=min_samples,
    metric="cosine",
    algorithm="brute",
    cluster_selection_method=artifact_run["cluster_selection_method"],
    n_jobs=HDBSCAN_N_JOBS,
    copy=True,
)
saved_model.fit(hybrid_matrices[config_name])
if not np.array_equal(saved_model.labels_, artifact_labels):
    raise RuntimeError(
        "Os labels do modelo refeito nao coincidem com os assignments do ARTIFACT_RUN_ID."
    )

encoder.save(str(model_output_dir / "sentence_transformer"))
pd.to_pickle(saved_model, model_output_dir / "hdbscan_model.pkl")
pd.to_pickle(vectorizers[config_name], model_output_dir / "tfidf_vectorizer.pkl")
assignments.to_csv(
    model_output_dir / "training_assignments.csv", index=False, encoding="utf-8-sig"
)
model_config = {
    "run_id": ARTIFACT_RUN_ID,
    "period": PERIOD_LABEL,
    "dataset_name": DATASET_NAME,
    "sample_seed": SAMPLE_RANDOM_STATE,
    "representation": "tfidf_mpnet_hybrid",
    "tfidf": {
        "config_name": config_name,
        "analyzer": artifact_run["analyzer"],
        "ngram_range": artifact_run["ngram_range"],
        "min_df": int(artifact_run["min_df"]),
        "max_features": MAX_FEATURES, "max_df": MAX_DF,
        "sublinear_tf": True,
        "norm": "l2",
        "vectorizer_artifact": "tfidf_vectorizer.pkl",
    },
    "encoder": {
        "model_name": MODEL_NAME,
        "artifact": "sentence_transformer/",
        "encode_batch_size": ENCODE_BATCH_SIZE,
        "normalize_embeddings_before_combination": True,
        "embedding_dimension": int(mpnet_matrix.shape[1]),
    },
    "combination": {
        "alpha": ALPHA,
        "normalization": "l2 per block",
        "weights": {
            "tfidf": float(np.sqrt(ALPHA)),
            "mpnet": float(np.sqrt(1.0 - ALPHA)),
        },
    },
    "hdbscan": {
        "min_cluster_size": int(artifact_run["min_cluster_size"]),
        "min_samples": min_samples,
        "metric": "cosine",
        "algorithm": "brute",
        "cluster_selection_method": artifact_run["cluster_selection_method"],
        "artifact": "hdbscan_model.pkl",
    },
    "frequency_weighted": False,
    "assignments_csv": str(assignment_path.relative_to(PROJECT_ROOT)),
    "training_assignments": "training_assignments.csv",
    "prediction_note": (
        "HDBSCAN foi ajustado sem prediction_data; o modelo nao atribui diretamente "
        "novas descricoes a clusters."
    ),
}
(model_output_dir / "config.json").write_text(
    json.dumps(model_config, indent=2, ensure_ascii=False), encoding="utf-8"
)
logger.info("Assignments salvos em %s", assignment_path.relative_to(PROJECT_ROOT))
logger.info("Modelo salvo em %s", model_output_dir.relative_to(PROJECT_ROOT))


## 9. Interpretacao e limites

- O ranking filtra por quantidade minima e maxima de clusters e, por padrao, exige silhouette nao nula e maior que a fracao de ruido. Desative `RANKING_FILTER_SILHOUETTE_ABOVE_NOISE` para remover essa segunda regra.
- Cada descricao unica tem o mesmo peso no HDBSCAN; frequencia aparece nos relatorios.
- A silhouette exclui ruido e usa ate 20.000 itens. Compare-a com cobertura e quantidade de clusters.
- t-SNE e uma projecao exploratoria; o HTML 3D permite girar e aproximar a nuvem de pontos.
- HDBSCAN e salvo sem mecanismo de predicao para novos pontos; o encoder e o vetorizador ficam disponiveis para reproduzir a representacao de treino.
- Os clusters sao exploratorios, nao categorias fiscais validadas.
